In [1]:
!pip install -q transformers datasets accelerate fastembed scikit-learn

In [2]:
from google.colab import files
files.upload()

Saving contradiction_eval.csv to contradiction_eval.csv


{'contradiction_eval.csv': b'decision,expected,label,new,old,relation,similarity\r\nreplace,replace,updates,I just moved to Mumbai last month.,I live in Delhi and work as a data analyst.,updates,0.503\r\nreplace,replace,updates,I quit Infosys and joined Google last week.,I work at Infosys.,updates,0.715\r\nreplace,replace,updates,The exam got postponed to next Tuesday.,My exam is on Friday.,updates,0.537\r\nkeep,replace,updates,I got engaged yesterday!,I\'m single.,(gated out),0.345\r\nreplace,replace,updates,I sold my Civic and bought a Tesla.,I drive a Honda Civic.,updates,0.688\r\nreplace,replace,updates,I started eating meat again after five years.,I\'m vegetarian.,updates,0.552\r\nreplace,replace,updates,I switched to a Pixel 8 last week.,My phone is an iPhone 12.,updates,0.404\r\nreplace,replace,updates,I dropped out of VIT and I\'m working full time now.,I\'m studying computer science at VIT.,updates,0.579\r\nreplace,replace,updates,The team meeting has been moved to 5 PM.,The t

In [3]:
import random
import numpy as np
import pandas as pd
from datasets import load_dataset

random.seed(0)

CITIES = ["Jaipur", "Lucknow", "Kochi", "Nagpur", "Indore", "Pune", "Chandigarh", "Bhopal", "Surat", "Mysore", "Kolkata", "Patna"]
COMPANIES = ["TCS", "Wipro", "Flipkart", "Zomato", "Swiggy", "Paytm", "Adobe", "Oracle", "Zoho", "Deloitte", "Amazon", "Accenture"]
EVENTS = ["interview", "flight", "dentist appointment", "project deadline", "workshop", "presentation", "viva", "lab test"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
ITEMS = [("scooter", "bicycle"), ("tablet", "laptop"), ("camera", "drone"), ("watch", "fitness band"), ("sofa", "bed"), ("bike", "car")]
THINGS = [("rent", "8000", "10000"), ("monthly budget", "20000", "30000"), ("gym fee", "1500", "2000"), ("internet plan", "399", "599")]
LIKES = ["pizza", "rainy weather", "horror movies", "early mornings", "football", "coffee", "chess", "long drives", "sushi", "dancing"]
SKILLS = ["French", "swimming", "painting", "coding in Rust", "yoga", "photography"]
PETS = [("rabbit", "Coco"), ("parrot", "Mango"), ("hamster", "Peanut"), ("turtle", "Shelly")]
EXAMS = ["GRE", "CAT", "IELTS", "driving test", "semester exam", "aptitude test"]
STATUS = [("I'm unemployed.", "I got a job offer and I start next week."), ("I'm a student.", "I graduated last month and now I work full time."),
          ("I'm living with my parents.", "I moved into my own flat."), ("I'm sick with a fever.", "I recovered and I'm back at work."),
          ("I'm on a break from running.", "I started running again every morning.")]


def synth(n_each=75):
    rep, keep = [], []
    for _ in range(n_each):
        a, b = random.sample(CITIES, 2)
        rep.append((f"I live in {a}.", random.choice([f"I just moved to {b}.", f"I relocated to {b} last month.", f"I shifted to {b} for good."])))
        c1, c2 = random.sample(COMPANIES, 2)
        rep.append((f"I work at {c1}.", random.choice([f"I left {c1} and started at {c2}.", f"I resigned from {c1} and joined {c2} this week."])))
        ev = random.choice(EVENTS); d1, d2 = random.sample(DAYS, 2)
        rep.append((f"My {ev} is on {d1}.", random.choice([f"The {ev} has been moved to {d2}.", f"They rescheduled my {ev} to {d2}.", f"My {ev} got postponed to {d2}."])))
        o, nw = random.choice(ITEMS)
        rep.append((f"I have a {o}.", random.choice([f"I sold my {o} and bought a {nw}.", f"I gave away my {o}, I use a {nw} now."])))
        t, v1, v2 = random.choice(THINGS)
        rep.append((f"My {t} is {v1} rupees.", f"My {t} is now {v2} rupees."))
        ex = random.choice(EXAMS)
        rep.append((f"I'm preparing for the {ex}.", random.choice([f"I took the {ex} yesterday.", f"I cleared the {ex} last week."])))
        old, new = random.choice(STATUS); rep.append((old, new))
        x = random.choice(LIKES)
        rep.append((f"I love {x}.", random.choice([f"I can't stand {x}.", f"I really dislike {x} now."])))
        rep.append((f"I never eat {random.choice(['fish', 'eggs', 'cheese'])}.", "I eat it almost every day."))
        rep.append((f"I'm {random.choice(['an early bird', 'a morning person'])}.", "I'm a night owl and I sleep very late."))
        # keep: elaboration, other person, unrelated
        p, name = random.choice(PETS)
        keep.append((f"I have a {p}.", random.choice([f"My {p} is called {name}.", f"{name}, my {p}, loves to play."])))
        s = random.choice(SKILLS)
        keep.append((f"I'm learning {s}.", random.choice([f"I practise {s} every evening.", f"I found a good {s} class nearby."])))
        keep.append((f"I like {x}.", random.choice([f"I enjoyed {x} a lot last weekend.", f"My friends like {x} too."])))
        a2, b2 = random.sample(CITIES, 2)
        keep.append((f"I live in {a2}.", random.choice([f"My friend lives in {b2}.", f"My cousin just moved to {b2}."])))
        c3, c4 = random.sample(COMPANIES, 2)
        keep.append((f"I work at {c3}.", random.choice([f"My brother works at {c4}.", f"My team at {c3} won an award."])))
        ev2 = random.choice(EVENTS)
        keep.append((f"My {ev2} is on {random.choice(DAYS)}.", random.choice([f"I'm nervous about my {ev2}.", f"I prepared well for the {ev2}."])))
        keep.append((f"I'm learning {random.choice(SKILLS)}.", f"I bought a {random.choice(ITEMS)[0]} yesterday."))
        keep.append((f"I live in {random.choice(CITIES)}.", f"My favourite food is {random.choice(LIKES)}."))
    return rep, keep


def build_train(use_synth, n_mnli=15000):
    ds = load_dataset("nyu-mll/multi_nli")["train"].to_pandas()
    ds = ds[ds.label.isin([0, 1, 2])]
    rep = ds[ds.label == 2].sample(n_mnli, random_state=0)
    keep = ds[ds.label != 2].sample(n_mnli, random_state=0)
    df = pd.concat([rep.assign(y=1), keep.assign(y=0)])[["premise", "hypothesis", "y"]]
    if use_synth:
        r, k = synth()
        s = pd.DataFrame([(a, b, 1) for a, b in r] + [(a, b, 0) for a, b in k], columns=["premise", "hypothesis", "y"])
        df = pd.concat([df, s])
        print(f"added {len(s)} template pairs ({len(r)} replace, {len(k)} keep)")
    return df.sample(frac=1, random_state=0).reset_index(drop=True)

print("data code ready")

data code ready


In [7]:
import torch, time
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

BASE = "distilroberta-base"
tok = AutoTokenizer.from_pretrained(BASE)


def train_model(df, out_dir, epochs=2):
    model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=2)
    ds = Dataset.from_pandas(df.rename(columns={"y": "labels"}))
    ds = ds.map(lambda b: tok(b["premise"], b["hypothesis"], truncation=True, max_length=96), batched=True)
    args = TrainingArguments(output_dir=out_dir, num_train_epochs=epochs, per_device_train_batch_size=32,
                             learning_rate=2e-5, weight_decay=0.01, fp16=torch.cuda.is_available(),
                             logging_steps=200, save_strategy="no", report_to="none", seed=0)
    trainer = Trainer(model=model, args=args, train_dataset=ds,
                      data_collator=DataCollatorWithPadding(tok))
    trainer.train()
    return model.eval().to("cuda" if torch.cuda.is_available() else "cpu")


@torch.no_grad()
def predict(model, olds, news):
    enc = tok(list(olds), list(news), truncation=True, max_length=96, padding=True, return_tensors="pt").to(model.device)
    return torch.softmax(model(**enc).logits, dim=-1)[:, 1].cpu().numpy()

print("training code ready, device:", "GPU" if torch.cuda.is_available() else "CPU (switch the runtime to GPU!)")

training code ready, device: GPU


In [5]:
import numpy as np, pandas as pd, time, torch
from fastembed import TextEmbedding

REPLACE = {"updates", "contradicts"}
GATE = 0.35
emb = TextEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")


def sims(olds, news):
    U = np.array(list(emb.embed(list(olds)))); V = np.array(list(emb.embed(list(news))))
    U /= np.linalg.norm(U, axis=1, keepdims=True); V /= np.linalg.norm(V, axis=1, keepdims=True)
    return (U * V).sum(axis=1)


def load_test(path):
    t = pd.read_csv(path, encoding="utf-8-sig")
    t = t.rename(columns={"old_text": "old", "new_text": "new"})
    return t


def evaluate(name, model, test, thr=0.5):
    olds, news = test["old"].tolist(), test["new"].tolist()
    t0 = time.time(); p = predict(model, olds, news); ms = (time.time() - t0) / len(test) * 1000
    sim = sims(olds, news)
    exp = test.label.isin(REPLACE).to_numpy()
    print(f"\n=========== {name} ===========  ({len(test)} pairs, {ms:.1f} ms per pair)")
    for title, dec in (("alone", p >= thr), (f"after gate {GATE}", (p >= thr) & (sim >= GATE))):
        tp = int((dec & exp).sum()); fp = int((dec & ~exp).sum()); fn = int((~dec & exp).sum()); tn = int((~dec & ~exp).sum())
        prec = tp / (tp + fp) if tp + fp else float("nan"); rec = tp / (tp + fn) if tp + fn else float("nan")
        print(f"{title:<16} correct replaces {tp}  WRONG replaces {fp}  missed {fn}  correct keeps {tn}  precision {prec:.2f}  recall {rec:.2f}")
        for lab in ("updates", "contradicts", "consistent", "unrelated"):
            m = (test.label == lab).to_numpy()
            if m.sum():
                ok = dec[m] if lab in REPLACE else ~dec[m]
                print(f"    {lab:<12} {int(ok.sum())}/{int(m.sum())}")
    gated = (p >= thr) & (sim >= GATE)
    for _, r in test[(gated != exp)].assign(p=p[(gated != exp)].round(2)).iterrows():
        print(f"  {'WRONG REPLACE' if not r.label in REPLACE else 'MISSED'} [{r.label} p={r.p}] {r.old!r} -> {r['new']!r}")
    if "decision" in test.columns:
        print("hybrid (classifier when confident, language model otherwise), gated pairs only:")
        g = sim >= GATE
        for lo, hi in ((0.2, 0.8), (0.1, 0.9), (0.3, 0.7)):
            llm = (test.decision == "replace").to_numpy()
            dec = np.where(p >= hi, True, np.where(p <= lo, False, llm)) & g
            calls = int((g & (p > lo) & (p < hi)).sum())
            fp = int((dec & ~exp).sum()); fn = int((~dec & exp).sum()); tp = int((dec & exp).sum())
            print(f"  confident if p<={lo} or p>={hi}: language model calls {calls}/{int(g.sum())}  wrong replaces {fp}  missed {fn}  recall {tp / max(exp.sum(), 1):.2f}")
    return p

print("evaluation code ready")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

evaluation code ready


In [8]:
test = load_test("contradiction_eval.csv")

# language model judge baseline from the saved decisions
exp = test.label.isin(REPLACE).to_numpy(); llm = (test.decision == "replace").to_numpy()
print(f"LANGUAGE MODEL JUDGE: correct replaces {int((llm & exp).sum())}  WRONG replaces {int((llm & ~exp).sum())}  missed {int((~llm & exp).sum())}")

model_a = train_model(build_train(False), "outA")
evaluate("A: MNLI only", model_a, test)
del model_a; torch.cuda.empty_cache()

model_b = train_model(build_train(True), "outB")
evaluate("B: MNLI plus template pairs", model_b, test)

LANGUAGE MODEL JUDGE: correct replaces 23  WRONG replaces 0  missed 2


Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Step,Training Loss
200,0.616459
400,0.468310
600,0.432557
800,0.431385
1000,0.386290
1200,0.327427
1400,0.326665
1600,0.335505
1800,0.326442


...

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/31350 [00:00<?, ? examples/s]

Step,Training Loss
200,0.621692
400,0.460547
600,0.434819
800,0.409229
1000,0.396820
1200,0.315428
1400,0.318516
1600,0.323053
1800,0.308200



=========== B: MNLI plus template pairs ===========  (60 pairs, 0.3 ms per pair)
alone            correct replaces 24  WRONG replaces 7  missed 1  correct keeps 28  precision 0.77  recall 0.96
    updates      14/15
    contradicts  10/10
    consistent   18/20
    unrelated    10/15
after gate 0.35  correct replaces 22  WRONG replaces 2  missed 3  correct keeps 33  precision 0.92  recall 0.88
    updates      13/15
    contradicts  9/10
    consistent   18/20
    unrelated    15/15
  MISSED [updates p=0.9599999785423279] "I'm single." -> 'I got engaged yesterday!'
  MISSED [updates p=0.09000000357627869] "I'm a junior developer." -> 'I got promoted to senior developer.'
  MISSED [contradicts p=0.5699999928474426] "I'm an only child." -> 'I have two older brothers.'
  WRONG REPLACE [consistent p=0.9700000286102295] 'I read a lot of science fiction.' -> 'I just finished reading Dune.'
  WRONG REPLACE [consistent p=0.9399999976158142] 'I have a meeting on Monday.' -> 'I have a dentist a

array([0.9956346 , 0.97726256, 0.9813107 , 0.9631959 , 0.9908213 ,
       0.9680808 , 0.93851966, 0.99458355, 0.9908213 , 0.99227446,
       0.979784  , 0.94905907, 0.9167827 , 0.98749614, 0.09073549,
       0.9885804 , 0.99268264, 0.6280368 , 0.99248135, 0.9959849 ,
       0.99444515, 0.9961306 , 0.5651112 , 0.9948803 , 0.99298817,
       0.02998651, 0.08351588, 0.01711089, 0.00583127, 0.01424075,
       0.01398261, 0.01430946, 0.02903564, 0.02052779, 0.01616836,
       0.02648094, 0.01283358, 0.04599103, 0.05915588, 0.0459482 ,
       0.11899801, 0.9701552 , 0.9433476 , 0.02010024, 0.253861  ,
       0.00486742, 0.02964751, 0.01652956, 0.6867378 , 0.20593424,
       0.01159736, 0.01566365, 0.56385833, 0.13488983, 0.8655654 ,
       0.6614857 , 0.53488207, 0.00948571, 0.05084657, 0.03182875],
      dtype=float32)

In [9]:
p = predict(model_b, test["old"].tolist(), test["new"].tolist())
sim = sims(test["old"].tolist(), test["new"].tolist())
g = sim >= GATE
exp = test.label.isin(REPLACE).to_numpy()
llm = (test.decision == "replace").to_numpy()

print(f"Language model alone: {int(g.sum())} calls on gated pairs, "
      f"wrong replaces {int((llm & g & ~exp).sum())}, missed {int((~(llm & g) & exp).sum())}")
for thr in (0.5, 0.3, 0.1):
    cand = g & (p >= thr)            # the classifier says replace (or maybe), so ask the language model
    dec = cand & llm
    print(f"Classifier screens at {thr}: calls {int(cand.sum())}/{int(g.sum())}  "
          f"correct replaces {int((dec & exp).sum())}  wrong replaces {int((dec & ~exp).sum())}  "
          f"missed {int((~dec & exp).sum())}  recall {(dec & exp).sum() / exp.sum():.2f}")

Language model alone: 42 calls on gated pairs, wrong replaces 0, missed 2
Classifier screens at 0.5: calls 24/42  correct replaces 22  wrong replaces 0  missed 3  recall 0.88
Classifier screens at 0.3: calls 24/42  correct replaces 22  wrong replaces 0  missed 3  recall 0.88
Classifier screens at 0.1: calls 25/42  correct replaces 22  wrong replaces 0  missed 3  recall 0.88
